In [ ]:
# ==========================================
# 🚀 Trash Detection YOLOv8 Training (Drive-safe)
# ==========================================

# 1️⃣ Install Ultralytics YOLOv8
!pip install --upgrade ultralytics
!pip install --upgrade --quiet gdown  # optional (for Drive downloads)
!pip install -U ultralytics albumentations
!pip install wandb
!wandb login



#It’ll show a link — click it → authorize → copy the API key → paste it in Colab.

# 2️⃣ Import Libraries
from ultralytics import YOLO
from google.colab import drive
import os
import albumentations as A
import wandb


# # 3️⃣ Mount Google Drive
drive.mount('/content/drive')



wandb: Currently logged in as: ishita03 (ishita03-university-of-pennsylvania) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!nvidia-smi

Fri Nov 14 02:49:25 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Current device:", torch.cuda.get_device_name(0))

CUDA available: True
Current device: Tesla T4


In [ ]:

# 4️⃣ Define dataset path (unzipped folder in Drive)
data_dir = '/content/drive/MyDrive/Garbagedata/Ishita_Model/Garbagedata'
data_yaml = os.path.join(data_dir, 'data_rs.yaml')

# ✅ Check that data.yaml exists
if not os.path.exists(data_yaml):
    raise FileNotFoundError(f"❌ data_rs.yaml not found in {data_dir}. Please check your dataset structure.")

print(f"✅ Dataset path set to: {data_dir}")
print(f"✅ Using config: {data_yaml}")


✅ Dataset path set to: /content/drive/MyDrive/Garbagedata/Ishita_Model/Garbagedata
✅ Using config: /content/drive/MyDrive/Garbagedata/Ishita_Model/Garbagedata/data_rs.yaml


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# custom_augmentations = A.Compose([
#     A.GaussianBlur(blur_limit=(3, 7), p=0.5),      # Motion blur / camera blur
#     A.ISONoise(intensity=(0.1, 0.5), p=0.5),       # Simulate sensor noise
#     # Add more augmentations if needed
# ])

In [ ]:
# Load model
model = YOLO('yolov8m.pt')

#Model Save Dir
save_dir = os.path.join(data_dir, 'Models_V3m_Runs')#change everytime you run this code keep incrementing cmpi by 1
os.makedirs(save_dir, exist_ok=True)

# Train
model.train(
    data=data_yaml,
    epochs=120,
    patience=25,
    cos_lr=True,
    lr0=0.002,          # ✅ custom learning rate
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    imgsz=512,
    batch=8,
    name='Model_V3m_Run1',
    project=save_dir,
    device='cuda',        # gpu
    save_period=5,
    exist_ok=True,
    # --- Augmentations ---
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    degrees=5,
    translate=0.05,
    scale=0.2,
    shear=2,
    perspective=0.0002,
    hsv_h=0.010,
    hsv_s=0.35,
    hsv_v=0.2,
    auto_augment="randaugment"
    #albumentations = custom_augmentations
  )

print(f"✅ Training started on METAL & PLASTIC subset — checkpoints saved in {save_dir}")


Ultralytics 8.3.228 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/Garbagedata/Ishita_Model/Garbagedata/data_rs.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=120, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.35, hsv_v=0.2, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=0.0, multi_scale=False, name=Model_V3m_Run1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pa

In [ ]:
results_val = model.val(data=data_yaml)
print("✅ Validation complete on METAL & PLASTIC subset!")

final_model_path = os.path.join(data_dir, "Model_V3m_Run1.pt")#increment fi
model.save(final_model_path)
print(f"✅ Final model saved at: {final_model_path}")


Ultralytics 8.3.228 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,918 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.9±0.3 ms, read: 7.5±4.5 MB/s, size: 14.7 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/1W5NeTCmXeHUuzwzy2YQC-K-G6uBNsKyw/Garbagedata/Ishita_Model/Garbagedata/val_mp_rs/labels.cache... 814 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 814/814 807.7Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 2.6it/s 19.7s
                   all        814       2778      0.744      0.566      0.625      0.435
                 METAL        464       1408      0.772      0.616      0.689      0.488
               PLASTIC        358       1370      0.716      0.516      0.561      0.382
Speed: 1.2ms preprocess, 13.5ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to /content/runs/detect/val
✅ Vali